# INTACT example

The main application scenarios of INTACT are presented as below.

### Application1: Temporal integration

2D example-HeLa

In [1]:
import INTACT as integ
import os
from importlib import reload
reload(integ)
if __name__ == "__main__":
    # Example usage 
    csv1 = os.path.join('data', 'HeLa', 'tracking_time1.csv')
    csv2 = os.path.join('data', 'HeLa', 'tracking_time2.csv')
    output = os.path.join('data', 'HeLa')
    
    movieInfo_new = integ.tracklets_integration(
        csv_path1=csv1,
        csv_path2=csv2,
        seg_indice = 'matlab',
        output_path=output,
        img_shape = [700, 1100],  
        spatial_shift = [[0,0],[0,0]],  
        temporal_shift = [0,31],
        mode = 'temporal',  
        solver = 'bb',   # 'pulp' or 'gurobi' or 'bb'
        saveSeg = False,
    )

Start Tracklet Integration!

Step 1: Reading tracking data...W
  Tracklet 1: 5162 detections
  Tracklet 2: 7521 detections
  Reading time: 3.25 seconds

Step 2: Selecting common cells...
  Selection time: 0.01 seconds

Step 3: Find overlapping nodes...
  Found 3169 matching pairs
  Matching time: 0.70 seconds

Step 4 & 5: Solving fusion...

Step 4-1 + Step 4-2a: Building easy arcs with C++...

Step 4-2b: Solving easy subgraphs with Python mcc4mot...

Step 4-3: Solving hard subgraphs...
  Fusion time: 160.15 seconds

Step 6: Creating fused tracklet...
  Fused tracklet has 9449 detections
 Fusion time: 0.03 seconds

Step 7: Start saving...
CSV file saved to: data/HeLa/tracking_result_merged.csv
  Saving time: 0.47 seconds

Fusion Complete!


3D example-CE

In [ ]:
# 3D
import INTACT as integ
import os
from importlib import reload
reload(integ)
if __name__ == "__main__":
    # Example usage
    csv1 = os.path.join('data', 'CE', 'tracking_time1.csv')
    csv2 = os.path.join('data', 'CE', 'tracking_time2.csv')
    output = os.path.join('data', 'CE')

    movieInfo_new = integ.tracklets_integration(
        csv_path1=csv1,
        csv_path2=csv2,
        seg_indice = 'matlab',
        output_path=output,
        resolution=[11.0, 1.0, 1.0],
        img_shape = [35,512,708],   
        spatial_shift = [[0,0,0],[0,0,0]], 
        temporal_shift = [0,83],
        mode = 'temporal',    
        solver = 'gurobi',   # 'pulp' or 'gurobi' or 'bb'
        saveSeg = False,
    )


Prepare fusion inputs from CSV
mode=temporal
csv_path1=data/HeLa/tracking_time1.csv
csv_path2=data/HeLa/tracking_time2.csv

Step 1: Reading tracking data...
  Tracklet 1: 5162 detections
  Tracklet 2: 7521 detections
  Reading time: 3.2758 s

Step 2: Selecting common cells...
  Partial tracklet 1: 3233 detections
  Partial tracklet 2: 3293 detections
  Selection time: 0.0072 s
  crop_size: [700, 1100]
  crop_area: []
  tm_shift: 0

Step 3: Finding overlapping nodes...
  Found 3169 matching pairs
  Matching time: 0.6995 s

Benchmark solve_fusion only
repeat=1, warmup=0
deepcopy_inputs=True
alternate_order=True
num movieInfo1_partial: 3233
num movieInfo2_partial: 3293
num matches: 3169

--- repeat 0, order=['old', 'new'] ---

Step 4-1: Building subgraphs...

Step 4-2: Solving easy subgraphs...

Step 4-3: Solving hard subgraphs...
old solve_fusion time: 177.3984 s

Step 4-1 + Step 4-2a: Building easy arcs with C++...

Step 4-2b: Solving easy subgraphs with Python mcc4mot...

Step 4-3: So

In [4]:
import INTACT as integ
import benchmark_fusion as bf
import os
import branchbound_old as old_fusion
import branchbound as new_fusion
from C_package import fast_easy_fusion_wrapper as easy_cpp
csv1 = os.path.join("data", "HeLa", "tracking_time1.csv")
csv2 = os.path.join("data", "HeLa", "tracking_time2.csv")
inputs = bf.prepare_fusion_inputs_from_csv(
    integ,
    csv_path1=csv1,
    csv_path2=csv2,
    seg_indice="matlab",
    img_shape=[700, 1100],
    spatial_shift=[[0, 0], [0, 0]],
    temporal_shift=[0, 31],
    mode="temporal",
)

easy_result = bf.benchmark_easy_part(
    inputs["movieInfo1_partial"],
    inputs["movieInfo2_partial"],
    inputs["matches"],
    old_solver=old_fusion,
    new_solver=easy_cpp, 
    repeat=1,
    deepcopy_inputs=True,
)

arc_debug = bf.compare_easy_arcs_debug(
    inputs["movieInfo1_partial"],
    inputs["movieInfo2_partial"],
    inputs["matches"],
    old_solver=old_fusion,
    new_solver=easy_cpp,
    decimals=4,
)


Prepare fusion inputs from CSV
mode=temporal
csv_path1=data/HeLa/tracking_time1.csv
csv_path2=data/HeLa/tracking_time2.csv

Step 1: Reading tracking data...
  Tracklet 1: 5162 detections
  Tracklet 2: 7521 detections
  Reading time: 3.3343 s

Step 2: Selecting common cells...
  Partial tracklet 1: 3233 detections
  Partial tracklet 2: 3293 detections
  Selection time: 0.0086 s
  crop_size: [700, 1100]
  crop_area: []
  tm_shift: 0

Step 3: Finding overlapping nodes...
  Found 3169 matching pairs
  Matching time: 0.6638 s

Benchmark Step 4-1 / 4-2 only

--- repeat 0 ---
old Step 4-1/4-2 time: 10.8233 s
new Step 4-1/4-2 time: 0.1240 s

===== Time: Step 4-1 / 4-2 =====
old times: [10.823285200167447]
new times: [0.1240269341506064]
old mean: 10.8233 s
new mean: 0.1240 s
speedup: 87.266x

===== Result comparison: tracks_easy =====
exact same tracks: False
num tracks old/new: 151 151
num selected cells old/new: 3229 3229
num trajectory edges old/new: 3078 3078
tracks only in old: 141
track

In [3]:
# 1. 用 C++ debug wrapper 只构造 arcs，不用 C++ solver
detection_cpp, transition_cpp, easy_subgraphs_cpp, hard_subgraphs_cpp, edges_cpp = (
    easy_cpp.build_easy_arcs_debug_cpp_wrapper(
        inputs["movieInfo1_partial"],
        inputs["movieInfo2_partial"],
        inputs["matches"],
    )
)

# 2. 把 C++ 构造的 arcs 交给旧版 Python mcc4mot 求解
traj_cpp_arcs_py_solver, cost_cpp_arcs_py_solver = old_fusion.mcc4mot(
    detection_cpp,
    transition_cpp,
)

N = (
    len(inputs["movieInfo1_partial"]["frames"])
    + len(inputs["movieInfo2_partial"]["frames"])
)

tracks_cpp_arcs_py_solver = [
    track[track < N]
    for track in traj_cpp_arcs_py_solver
]

# 3. 旧版 Python pruning 的结果
easy_subgraphs_old, hard_subgraphs_old, edges_old = old_fusion.build_subgraphs(
    inputs["movieInfo1_partial"],
    inputs["movieInfo2_partial"],
    inputs["matches"],
)

tracks_easy_old, costs_old = old_fusion.pruning(
    easy_subgraphs_old,
    inputs["movieInfo1_partial"],
    inputs["movieInfo2_partial"],
    edges_old,
)

# 4. 比较：旧版 pruning vs C++ arcs + Python solver
cmp_cpp_arcs_py_solver = bf.compare_tracks(
    tracks_easy_old,
    tracks_cpp_arcs_py_solver,
    name="old pruning vs cpp arcs + python mcc4mot",
)


===== Result comparison: old pruning vs cpp arcs + python mcc4mot =====
exact same tracks: False
num tracks old/new: 151 151
num selected cells old/new: 3229 3229
num trajectory edges old/new: 3078 3078
tracks only in old: 141
tracks only in new: 141
cells only in old: 1519
cells only in new: 1519
trajectory edges only in old: 2309
trajectory edges only in new: 2309

--- Preview: tracks only in old ---
(0, 3233, 3316, 247, 327, 3564, 3649, 581, 3817, 3907, 844, 937, 4186, 1132, 4387, 4492, 1448, 4710, 4821, 4935, 5050, 2012, 2130, 2249, 2369, 2492, 2611, 2735, 2857, 2982, 6272, 6400)
(1, 3235, 3315, 241, 3478, 3565, 497, 585, 3821, 3912, 4001, 4093, 4190, 4289, 4392, 1344, 1453, 1562, 1673, 1784, 1899, 2017, 5291, 5411, 2373, 5655, 5778, 2739, 6024, 2985, 3109, 6401)
(2, 3239, 163, 3398, 329, 416, 3656, 3741, 3824, 3916, 853, 4103, 1040, 4292, 1244, 4507, 1458, 4724, 4831, 4951, 1910, 5185, 5307, 2263, 2382, 2505, 2623, 2748, 6037, 2996, 3117, 6414)
(3, 3236, 166, 243, 326, 412, 494, 

### Application2: Spatial integration

2D example-Neu

In [ ]:
import INTACT as integ
import os
from importlib import reload
reload(integ)
if __name__ == "__main__":
    # Example usage
    csv1 = os.path.join('data', 'Neu', 'tracking_space1.csv')
    csv2 = os.path.join('data', 'Neu', 'tracking_space2.csv')
    output = os.path.join('data', 'Neu')
    
    movieInfo_new = integ.tracklets_integration(
        csv_path1=csv1,
        csv_path2=csv2,
        seg_indice = 'matlab',
        output_path=output,
        resolution=[1.0, 1.0],
        img_shape = [800,1280],   
        img_shape1 = [420,1280],
        img_shape2 = [420,1280],
        spatial_shift = [[0,0],[380,0]], 
        temporal_shift = [0,0],
        mode = 'spatial',  
        solver = 'pulp',     # 'pulp' or 'gurobi' or 'bb'
        saveSeg = False,
    )

3D example-TRIF

In [ ]:
import INTACT as integ
import os
from importlib import reload
reload(integ)
if __name__ == "__main__":
    # Example usage
    csv1 = os.path.join('data', 'TRIF', 'tracking_space1.csv')
    csv2 = os.path.join('data', 'TRIF', 'tracking_space2.csv')
    output = os.path.join('data', 'TRIF')

    movieInfo_new = integ.tracklets_integration(
        csv_path1=csv1,
        csv_path2=csv2,
        seg_indice = 'matlab',
        output_path=output,
        resolution=[1.0, 1.0, 1.0], 
        img_shape = [496,936,483],    
        img_shape1 = [496,483,483],
        img_shape2 = [496,483,483],
        spatial_shift = [[0,0,0],[0,453,0]], 
        temporal_shift = [0,0],
        mode = 'spatial',  
        solver = 'gurobi',      # 'pulp' or 'gurobi' or 'bb'
        saveSeg = False,
    )

### Application3: Multi-view stitching

In [ ]:
# multi-view
import INTACT as integ
from importlib import reload
reload(integ)
if __name__ == "__main__":
    # Example usage
    csv1 = os.path.join('data', 'FISH', 'movieInfoAll_0812_transformed.csv')
    csv2 = os.path.join('data', 'FISH', 'movieInfoAll_1014_transformed.csv')
    output = os.path.join('data', 'FISH') 

    movieInfo_new = integ.tracklets_integration(
        csv_path1 = csv1,
        csv_path2 = csv2,
        seg_indice = 'matlab',
        output_path=output,
        resolution=[4.546, 1.0, 1.0], 
        img_shape = [1837,1954,1928],    
        img_shape1 = [1837,1954,1928],
        img_shape2=[1837,1954,1928],
        spatial_shift = [[0,0,0],[0,0,0]], 
        temporal_shift = [0,0],
        mode = 'spatial',    
        solver = 'gurobi',     # 'pulp' or 'gurobi' or 'bb'
        saveSeg = False,
    )